In [5]:
import pandas as pd

df_monthly = pd.read_csv("Monthly_Rainfall_Rathnapura.csv")
df_days = pd.read_csv("Annual_Rain_Inference_Days_Rathnapura.csv")

print("=== Monthly Rainfall Columns ===")
print(df_monthly.columns.tolist())
print("\n=== Rainy Days Columns ===")
print(df_days.columns.tolist())

print("\n=== Monthly Rainfall First 3 Rows ===")
print(df_monthly.head(3))

print("\n=== Rainy Days All Rows ===")
print(df_days)

=== Monthly Rainfall Columns ===
['Date', 'Rainfall_Rathnapura']

=== Rainy Days Columns ===
['Year', 'No of Rainy Days_ Rathnapura']

=== Monthly Rainfall First 3 Rows ===
      Date Rainfall_Rathnapura
0  1995-01                 195
1  1995-02                  84
2  1995-03                 131

=== Rainy Days All Rows ===
        Year  No of Rainy Days_ Rathnapura
0  1961-1990                           205
1  1991-2010                           223
2  2011-2020                           225
3       2021                           198
4       2022                           226
5       2023                           244
6       2024                           236


In [9]:
import numpy as np
import pandas as pd
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# ==========================================
# 1. LOAD YOUR TWO CSV FILES
# ==========================================
df_monthly = pd.read_csv("Monthly_Rainfall_Rathnapura.csv")
df_monthly['Date'] = pd.to_datetime(df_monthly['Date'], format='%Y-%m', errors='coerce')
df_monthly = df_monthly.dropna(subset=['Date']).reset_index(drop=True)
df_monthly['Year'] = df_monthly['Date'].dt.year.astype(int)
df_monthly['Rainfall_Rathnapura'] = pd.to_numeric(df_monthly['Rainfall_Rathnapura'], errors='coerce')
df_monthly = df_monthly.dropna(subset=['Rainfall_Rathnapura']).reset_index(drop=True)

# Annual rain interference days CSV
df_days = pd.read_csv("Annual_Rain_Inference_Days_Rathnapura.csv")
RAINY_COL = 'No of Rainy Days_ Rathnapura'
df_days[RAINY_COL] = pd.to_numeric(df_days[RAINY_COL], errors='coerce')

# ==========================================
# 2. FUZZY SYSTEM (same as your methodology)
# ==========================================
rainfall_annual = ctrl.Antecedent(np.arange(0, 5001, 1), 'rainfall_annual')
rain_interf_days = ctrl.Antecedent(np.arange(0, 366, 1), 'rain_interf_days')
beta_out = ctrl.Consequent(np.arange(1.00, 1.41, 0.01), 'beta', defuzzify_method='centroid')

rainfall_annual['low'] = fuzz.trapmf(rainfall_annual.universe, [0, 0, 1000, 1750])
rainfall_annual['medium'] = fuzz.trapmf(rainfall_annual.universe, [1000, 1750, 2500, 3000])
rainfall_annual['high'] = fuzz.trapmf(rainfall_annual.universe, [2500, 3000, 5000, 5000])

rain_interf_days['low'] = fuzz.trapmf(rain_interf_days.universe, [0, 0, 30, 60])
rain_interf_days['medium'] = fuzz.trapmf(rain_interf_days.universe, [30, 60, 90, 150])
rain_interf_days['high'] = fuzz.trapmf(rain_interf_days.universe, [90, 150, 365, 365])

beta_out['no_loading'] = fuzz.trimf(beta_out.universe, [1.00, 1.00, 1.10])
beta_out['low_loading'] = fuzz.trimf(beta_out.universe, [1.00, 1.10, 1.20])
beta_out['mod_loading'] = fuzz.trimf(beta_out.universe, [1.10, 1.20, 1.30])
beta_out['high_loading'] = fuzz.trimf(beta_out.universe, [1.20, 1.30, 1.38])
beta_out['vhigh_loading'] = fuzz.trimf(beta_out.universe, [1.30, 1.38, 1.38])

rule1 = ctrl.Rule(rainfall_annual['low'] & rain_interf_days['low'], beta_out['low_loading'])
rule2 = ctrl.Rule(rainfall_annual['low'] & rain_interf_days['medium'], beta_out['mod_loading'])
rule3 = ctrl.Rule(rainfall_annual['low'] & rain_interf_days['high'], beta_out['mod_loading'])
rule4 = ctrl.Rule(rainfall_annual['medium'] & rain_interf_days['low'], beta_out['no_loading'])
rule5 = ctrl.Rule(rainfall_annual['medium'] & rain_interf_days['medium'], beta_out['mod_loading'])
rule6 = ctrl.Rule(rainfall_annual['medium'] & rain_interf_days['high'], beta_out['high_loading'])
rule7 = ctrl.Rule(rainfall_annual['high'] & rain_interf_days['low'], beta_out['low_loading'])
rule8 = ctrl.Rule(rainfall_annual['high'] & rain_interf_days['medium'], beta_out['high_loading'])
rule9 = ctrl.Rule(rainfall_annual['high'] & rain_interf_days['high'], beta_out['vhigh_loading'])

beta_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9])
beta_sim = ctrl.ControlSystemSimulation(beta_ctrl)

# ==========================================
# 3. ROLLING EXPECTED β FOR EACH PURCHASE YEAR
# ==========================================
purchase_years = [2022, 2023, 2024, 2025]
print("=" * 80)
print("ROLLING EXPECTED β FOR EACH PURCHASE YEAR")
print("=" * 80)

for purchase_year in purchase_years:
    end_year = purchase_year - 1                    # data up to previous year only
    # Filter monthly data
    df_hist = df_monthly[df_monthly['Year'] <= end_year].copy()
    annual_rain = df_hist.groupby('Year')['Rainfall_Rathnapura'].sum().reset_index()
    
    # Assign rainy days (period averages + exact)
    rainy_days_dict = {}
    rainy_days_dict.update({y: 223 for y in range(1995, 2011)})
    rainy_days_dict.update({y: 225 for y in range(2011, 2021)})
    for _, row in df_days.iterrows():
        year_str = str(row['Year']).strip()
        if '-' not in year_str:
            try:
                y = int(float(year_str))
                if y <= end_year and y in [2021, 2022, 2023, 2024]:
                    rainy_days_dict[y] = int(row[RAINY_COL])
            except:
                pass
    
    years = annual_rain['Year'].tolist()
    rainfalls = annual_rain['Rainfall_Rathnapura'].tolist()
    rainy_days_list = [rainy_days_dict.get(y, 223) for y in years]
    
    # Bootstrap Monte Carlo for this purchase year
    SIMS = 10000
    SEED = 42
    rng = np.random.default_rng(SEED)
    beta_samples = []
    for _ in range(SIMS):
        idx = rng.integers(0, len(years))
        rain = float(rainfalls[idx])
        days = float(rainy_days_list[idx])
        try:
            beta_sim.input['rainfall_annual'] = rain
            beta_sim.input['rain_interf_days'] = days
            beta_sim.compute()
            beta_samples.append(beta_sim.output['beta'])
        except:
            pass
    
    expected_beta = np.mean(beta_samples)
    
    print(f"Purchase year {purchase_year} → Expected β (using data 1995–{end_year}) = {expected_beta:.4f}")

print("=" * 80)

ROLLING EXPECTED β FOR EACH PURCHASE YEAR
Purchase year 2022 → Expected β (using data 1995–2021) = 1.3353
Purchase year 2023 → Expected β (using data 1995–2022) = 1.3360
Purchase year 2024 → Expected β (using data 1995–2023) = 1.3365
Purchase year 2025 → Expected β (using data 1995–2024) = 1.3370


In [15]:
import numpy as np
import pandas as pd
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# ================== INPUT ==================
purchase_year = 2025          # <--- Change this if you want another year
# ==========================================

df_monthly = pd.read_csv("Monthly_Rainfall_Rathnapura.csv")
df_monthly.columns = df_monthly.columns.str.strip().str.replace('\ufeff', '', regex=False)
df_monthly['Date'] = pd.to_datetime(df_monthly['Date'], format='%Y-%m', errors='coerce')
df_monthly = df_monthly.dropna(subset=['Date']).reset_index(drop=True)
df_monthly['Year'] = df_monthly['Date'].dt.year.astype(int)
df_monthly['Rainfall_Rathnapura'] = pd.to_numeric(df_monthly['Rainfall_Rathnapura'], errors='coerce')
df_monthly = df_monthly.dropna(subset=['Rainfall_Rathnapura']).reset_index(drop=True)

df_days = pd.read_csv("Annual_Rain_Inference_Days_Rathnapura.csv")
df_days.columns = df_days.columns.str.strip().str.replace('\ufeff', '', regex=False)
RAINY_COL = 'No of Rainy Days_ Rathnapura'
df_days[RAINY_COL] = pd.to_numeric(df_days[RAINY_COL], errors='coerce')

# Filter data up to previous year only
end_year = purchase_year - 1
df_hist = df_monthly[df_monthly['Year'] <= end_year].copy()

annual_rain = df_hist.groupby('Year')['Rainfall_Rathnapura'].sum().reset_index()

# Assign rainy days
rainy_days_dict = {}
rainy_days_dict.update({y: 223 for y in range(1995, 2011)})
rainy_days_dict.update({y: 225 for y in range(2011, 2021)})
for _, row in df_days.iterrows():
    year_str = str(row['Year']).strip()
    if '-' not in year_str:
        try:
            y = int(float(year_str))
            if y <= end_year:
                rainy_days_dict[y] = int(row[RAINY_COL])
        except:
            pass

years = annual_rain['Year'].tolist()
rainfalls = annual_rain['Rainfall_Rathnapura'].tolist()
rainy_days_list = [rainy_days_dict.get(y, 223) for y in years]

# Fuzzy system
rainfall_annual = ctrl.Antecedent(np.arange(0, 5001, 1), 'rainfall_annual')
rain_interf_days = ctrl.Antecedent(np.arange(0, 366, 1), 'rain_interf_days')
beta_out = ctrl.Consequent(np.arange(1.00, 1.41, 0.01), 'beta', defuzzify_method='centroid')

rainfall_annual['low']    = fuzz.trapmf(rainfall_annual.universe, [0, 0, 1000, 1750])
rainfall_annual['medium'] = fuzz.trapmf(rainfall_annual.universe, [1000, 1750, 2500, 3000])
rainfall_annual['high']   = fuzz.trapmf(rainfall_annual.universe, [2500, 3000, 5000, 5000])

rain_interf_days['low']    = fuzz.trapmf(rain_interf_days.universe, [0, 0, 30, 60])
rain_interf_days['medium'] = fuzz.trapmf(rain_interf_days.universe, [30, 60, 90, 150])
rain_interf_days['high']   = fuzz.trapmf(rain_interf_days.universe, [90, 150, 365, 365])

beta_out['no_loading']    = fuzz.trimf(beta_out.universe, [1.00, 1.00, 1.10])
beta_out['low_loading']   = fuzz.trimf(beta_out.universe, [1.00, 1.10, 1.20])
beta_out['mod_loading']   = fuzz.trimf(beta_out.universe, [1.10, 1.20, 1.30])
beta_out['high_loading']  = fuzz.trimf(beta_out.universe, [1.20, 1.30, 1.38])
beta_out['vhigh_loading'] = fuzz.trimf(beta_out.universe, [1.30, 1.38, 1.38])

rule1 = ctrl.Rule(rainfall_annual['low']    & rain_interf_days['low'],    beta_out['low_loading'])
rule2 = ctrl.Rule(rainfall_annual['low']    & rain_interf_days['medium'], beta_out['mod_loading'])
rule3 = ctrl.Rule(rainfall_annual['low']    & rain_interf_days['high'],   beta_out['mod_loading'])
rule4 = ctrl.Rule(rainfall_annual['medium'] & rain_interf_days['low'],    beta_out['no_loading'])
rule5 = ctrl.Rule(rainfall_annual['medium'] & rain_interf_days['medium'], beta_out['mod_loading'])
rule6 = ctrl.Rule(rainfall_annual['medium'] & rain_interf_days['high'],   beta_out['high_loading'])
rule7 = ctrl.Rule(rainfall_annual['high']   & rain_interf_days['low'],    beta_out['low_loading'])
rule8 = ctrl.Rule(rainfall_annual['high']   & rain_interf_days['medium'], beta_out['high_loading'])
rule9 = ctrl.Rule(rainfall_annual['high']   & rain_interf_days['high'],   beta_out['vhigh_loading'])

beta_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9])
beta_sim = ctrl.ControlSystemSimulation(beta_ctrl)

# Bootstrap Monte Carlo
SIMS = 10000
SEED = 42
rng = np.random.default_rng(SEED)
beta_samples = []
for _ in range(SIMS):
    idx = rng.integers(0, len(years))
    rain = float(rainfalls[idx])
    days = float(rainy_days_list[idx])
    beta_sim.input['rainfall_annual'] = rain
    beta_sim.input['rain_interf_days'] = days
    beta_sim.compute()
    beta_samples.append(beta_sim.output['beta'])

expected_beta = np.mean(beta_samples)
print(f"Expected β for purchase year {purchase_year} (using data up to {end_year}) = {expected_beta:.4f}")

Expected β for purchase year 2025 (using data up to 2024) = 1.3370
